``

In [7]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
import kagglehub
adhamashraf202200953_technical_parsed_questions_path = kagglehub.dataset_download('adhamashraf202200953/technical-parsed-questions')

print('Data source import complete.')

Using Colab cache for faster access to the 'technical-parsed-questions' dataset.
Data source import complete.


In [ ]:
# # Upgrade PyTorch to a unified CUDA 12.1 version
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# # Install and upgrade all Hugging Face and evaluation libraries together
# !pip install --upgrade transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score bert_score nltk

In [9]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [10]:
import gc
import torch
import os
import glob
import re
import pandas as pd
from datasets import Dataset
from transformers import (
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

In [11]:
# 2. DEFINING PREPROCESSING (WITH CLEANING FOR DOUBLE PREFIXES "A) A)")
def preprocess_function(examples):
    full_texts = []

    for i in range(len(examples["context"])):
        c = examples["context"][i]
        s = examples["scenario_context"][i]
        q = examples["question"][i]

        c0 = re.sub(r'^[A-Da-d][\s\)\.\-]*', '', str(examples["choices/0"][i])).strip()
        c1 = re.sub(r'^[A-Da-d][\s\)\.\-]*', '', str(examples["choices/1"][i])).strip()
        c2 = re.sub(r'^[A-Da-d][\s\)\.\-]*', '', str(examples["choices/2"][i])).strip()
        c3 = re.sub(r'^[A-Da-d][\s\)\.\-]*', '', str(examples["choices/3"][i])).strip()

        ans_match = re.search(r'[A-D]', str(examples["correct_choice"][i]))
        ans = ans_match.group(0) if ans_match else str(examples["correct_choice"][i]).strip()

        text = (
            f"Context: {str(c).strip()}\n\n"
            f"Scenario: {str(s).strip()}\n"
            f"Question: {str(q).strip()}\n"
            f"Choices:\n"
            f"A) {c0}\n"
            f"B) {c1}\n"
            f"C) {c2}\n"
            f"D) {c3}\n"
            f"Answer: {ans}{tokenizer.eos_token}"
        )
        full_texts.append(text)

    model_inputs = tokenizer(full_texts, max_length=MAX_LENGTH, truncation=True)
    return model_inputs

In [12]:
# 1. SET MODEL VARIABLES FOR LLAMA 3.2
MODEL_NAME = "unsloth/Llama-3.2-1B"
DATA_FOLDER = "/root/.cache/kagglehub/datasets/adhamashraf202200953/technical-parsed-questions/versions/5"
OUTPUT_DIR = "./outputs/STABLE_LLAMA_MCQ"

BATCH_SIZE = 2
GRADIENT_ACC_STEPS = 16
MAX_LENGTH = 512

gc.collect()
torch.cuda.empty_cache()

csv_files = glob.glob(os.path.join(DATA_FOLDER, "*_mcq.csv"))
if not csv_files:
    raise FileNotFoundError(f"No MCQ CSV files found in {DATA_FOLDER}")

required_columns = [
    "context", "scenario_context", "question",
    "choices/0", "choices/1", "choices/2", "choices/3",
    "correct_choice"
]

all_dfs = []
for file_path in csv_files:
    try:
        df = pd.read_csv(file_path, nrows=2)

        if not all(col in df.columns for col in required_columns):
            print(f"Skipping corrupt or mismatched file: {os.path.basename(file_path)}")
            continue

        df_clean = pd.read_csv(file_path, usecols=required_columns)
        all_dfs.append(df_clean)
        print(f"Successfully loaded {os.path.basename(file_path)} ({len(df_clean)} rows)")
    except Exception as e:
        print(f"Error reading {os.path.basename(file_path)}: {e}")

if not all_dfs:
    raise ValueError("No valid MCQ files could be loaded with the required columns.")

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df = combined_df.dropna(subset=["context", "question", "correct_choice"])

dataset = Dataset.from_pandas(combined_df)
print(f"\nFinal unified dataset size: {len(dataset)} samples.")

Successfully loaded SWE_Abstract_Theory_mcq.csv (761 rows)
Skipping corrupt or mismatched file: DSA_Abstract_Theory_mcq.csv
Successfully loaded DSA_Applied_Enginering_mcq.csv (732 rows)
Successfully loaded ML_DS_Abstract_Theory_mcq.csv (715 rows)

Final unified dataset size: 2208 samples.


In [13]:
# 4. TOKENIZER SETUP SPECIFIC FOR LLAMA 3.2
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

# 5. LOADING LLAMA MODEL AND PEFT CONFIG
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)

print("Total training samples:", len(split_dataset["train"]))

Map:   0%|          | 0/2208 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Total training samples: 1987


In [14]:
# 6. TRAINING ARGUMENTS
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=5,
    learning_rate=2e-4,
    num_train_epochs=3,
    max_grad_norm=1.0,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

Step,Training Loss,Validation Loss
50,1.694000,1.651558
100,1.569575,1.604637
150,1.497281,1.588682
189,1.518825,1.582955


TrainOutput(global_step=189, training_loss=1.6351784196480241, metrics={'train_runtime': 1384.4247, 'train_samples_per_second': 4.306, 'train_steps_per_second': 0.137, 'total_flos': 1.6057490766114816e+16, 'train_loss': 1.6351784196480241, 'epoch': 3.0})

In [15]:
# 7. ZIPPING THE ADAPTERS FOR DOWNLOAD
!zip -r "/content/LLAMA-3.2-1B-MCQ.zip" "/content/outputs"

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/STABLE_LLAMA_MCQ/ (stored 0%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/ (stored 0%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/README.md (deflated 65%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/optimizer.pt (deflated 9%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/scheduler.pt (deflated 62%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/tokenizer.json (deflated 85%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/training_args.bin (deflated 54%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/adapter_model.safetensors (deflated 8%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/trainer_state.json (deflated 75%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/tokenizer_config.json (deflated 45%)
  adding: content/outputs/STABLE_LLAMA_MCQ/checkpoint-189/rng_state.pth (deflated 26%)
  adding: content/outputs/STA

In [16]:
# ==================== INFERENCE SECTOR ====================

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
import torch

BASE_MODEL = "unsloth/Llama-3.2-1B"
LLAMA_LORA_DIR = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))[-1] if glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")) else OUTPUT_DIR

print(f"Loading Trained Llama MCQ Model from {LLAMA_LORA_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, LLAMA_LORA_DIR)

class StopOnEosCriteria(StoppingCriteria):
    def __init__(self, eos_token_id):
        self.eos_token_id = eos_token_id

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        return input_ids[0, -1].item() == self.eos_token_id

def ask_llama_mcq(context_text):
    prompt = f"Context: {context_text}\n\nScenario:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    stopping_criteria = StoppingCriteriaList([StopOnEosCriteria(tokenizer.eos_token_id)])

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        stopping_criteria=stopping_criteria
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_output = full_text[len(prompt):].strip()
    return f"Scenario: {generated_output}"

# 8. TESTING THE LLAMA MODEL
sample_context = """
Pillar: DSA/Applied_Enginering # Source_Book: Java3e20110103
Quicksort is inherently recursive, because each Quicksort operation must sort two sublists.
Implementing it iteratively requires managing a manual stack structure.
"""
print("\n--- Model Output Test ---")
print(ask_llama_mcq(sample_context))

# 9. DOWNLOAD THE COMPRESSED WEIGHTS TO YOUR LOCAL MACHINE
from google.colab import files
files.download("/content/LLAMA-3.2-1B-MCQ.zip")

Loading Trained Llama MCQ Model from ./outputs/STABLE_LLAMA_MCQ/checkpoint-189...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Model Output Test ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Scenario: You are building a high-performance sorting library for a large-scale data processing pipeline. The system must handle massive datasets and provide fast sorting capabilities.
Question: Why is it impractical to implement Quicksort as an iterative algorithm, given its inherent recursive nature?
Choices:
A) The overhead of recursive function calls makes it too slow for large datasets.
B) The manual stack structure required by Quicksort operations would lead to stack overflows and memory issues.
C) Iterative implementations are inherently slower than recursive ones due to the need for manual stack management.
D) Recursive function calls are not thread-safe and would cause concurrent sorting issues.
Answer: B


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>